# Multi-Agent AI Deep Researcher — v3 (Hackathon-Ready)

Nine-agent research system with conditional loops, persistent memory, multi-source RAG, multimodal analysis, source-trust scoring, and chart generation.

## Architecture

```
                    ┌─────────┐
START ────────────► │ Planner │
                    └────┬────┘
                         ▼
                ┌────────────────┐
            ┌──►│   Retriever    │◄──────┐
            │   └────────┬───────┘       │
            │            ▼               │
            │   ┌─────────────────┐      │
            │   │SourceReliability│      │  (loop fires
            │   └────────┬────────┘      │   when Reflection
            │            ▼               │   finds gaps or
            │   ┌────────────────┐       │   contradictions)
            │   │   Multimodal   │       │
            │   └────────┬───────┘       │
            │            ▼               │
            │   ┌────────────────┐       │
            │   │    Analyzer    │       │
            │   └────────┬───────┘       │
            │            ▼               │
            │   ┌────────────────┐       │
            │   │  FactChecker   │ (RAG) │
            │   └────────┬───────┘       │
            │            ▼               │
            │   ┌────────────────┐       │
            └───┤   Reflection   ├───────┘
                └────────┬───────┘
                         ▼ (when satisfied)
                ┌────────────────┐
                │  Visualization │
                └────────┬───────┘
                         ▼
                ┌────────────────┐
                │ ReportBuilder  │
                └────────┬───────┘
                         ▼
                        END
```

## What's wired tight (just runs)
- LangGraph with conditional routing + Reflection loop
- Persistent ChromaDB RAG (`./chroma_db`)
- Memory via SqliteSaver checkpointer (`./checkpoints.db`)
- Tavily + ArXiv + Wikipedia retrieval
- Multimodal vision via OpenRouter
- Pydantic structured output throughout
- `app.stream()` per-node visibility

## What's left as your team's slot (search "YOUR TODO")
- Trust criteria customization for your domain
- Prompt tuning for the vertical you pick
- Streamlit UI polish (separate `app.py` provided as starter)
- Custom agents (Critic / Citation / your idea)
- Tests
- README + final demo query

If everything were already implemented, the judges would notice. The TODO hooks are where your team's identity shows up.


## 1. Install dependencies

In [ ]:
!pip install -q uv

!uv pip install --system -q \
    langgraph \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    fastembed \
    tavily-python \
    arxiv \
    wikipedia \
    pypdf \
    pillow \
    pydantic \
    matplotlib \
    rich \
    grandalf

print("✅ Dependencies installed")


## 2. API keys (only two needed)

In [ ]:
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API Key: ")
os.environ["TAVILY_API_KEY"] = getpass("Tavily API Key: ")

print("✅ Keys set")


## 3. Imports

In [ ]:
import base64
import io
from typing import TypedDict, List, Annotated, Literal, Optional
from operator import add
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.sqlite import SqliteSaver

import matplotlib.pyplot as plt
from rich.console import Console
from rich.markdown import Markdown

console = Console()
print("✅ Imports loaded")


## 4. LLM tier setup (OpenRouter)

In [ ]:
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_HEADERS = {
    "HTTP-Referer": "https://hackathon.local",
    "X-Title": "Deep Researcher Hackathon",
}

# YOUR TODO: swap models if you want — see https://openrouter.ai/models
reasoning_llm = ChatOpenAI(
    model="anthropic/claude-3.5-sonnet",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=OPENROUTER_BASE, temperature=0.2,
    default_headers=OPENROUTER_HEADERS,
)

fast_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=OPENROUTER_BASE, temperature=0.1,
    default_headers=OPENROUTER_HEADERS,
)

vision_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  # has vision support
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base=OPENROUTER_BASE, temperature=0.2,
    default_headers=OPENROUTER_HEADERS,
)

print(reasoning_llm.invoke("Say 'ready'.").content)


## 5. Pydantic schemas

All structured outputs flow through these. Notice we have schemas for source trust, reflection (richer than basic gap analysis), and chart specs.

In [ ]:
class SubQuestion(BaseModel):
    question: str
    rationale: str

class ResearchPlan(BaseModel):
    sub_questions: List[SubQuestion]
    key_concepts: List[str]

class SourceTrust(BaseModel):
    source_id: str
    trust_score: Literal["high", "medium", "low"]
    rationale: str

class SourceAssessment(BaseModel):
    assessments: List[SourceTrust]

class FactCheckResult(BaseModel):
    claim: str
    verdict: Literal["supported", "unsupported", "contradicted"]
    evidence: str

class ReflectionResult(BaseModel):
    coverage_gaps: List[str] = Field(description="Sub-questions still under-supported")
    contradictions: List[str] = Field(description="Claims that disagree across sources")
    overlooked_angles: List[str] = Field(description="Important perspectives not yet researched")
    needs_more_research: bool

class ChartSpec(BaseModel):
    chart_type: Literal["bar", "line", "pie", "none"]
    title: str
    x_labels: List[str]
    y_values: List[float]
    y_label: str = ""


## 6. Persistent RAG vector store

`persist_directory` makes ChromaDB survive notebook restarts — your research corpus accumulates.

In [ ]:
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vector_store = Chroma(
    collection_name="research_corpus",
    embedding_function=embeddings,
    persist_directory="./chroma_db",   # persists across runs
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print("✅ ChromaDB ready (persistent)")


## 7. External retrieval tools

In [ ]:
tavily_tool = TavilySearchResults(max_results=4)
arxiv_tool = ArxivQueryRun(api_wrapper=ArxivAPIWrapper(top_k_results=2, doc_content_chars_max=2000))
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500))

print("✅ Tavily, ArXiv, Wikipedia ready")


## 8. Graph state

Extended from v2 with `source_assessments`, `reflection`, and `chart_path`.

In [ ]:
class ResearchState(TypedDict):
    query: str
    plan: Optional[ResearchPlan]
    raw_findings: Annotated[List[str], add]
    documents: Annotated[List[Document], add]
    source_assessments: List[SourceTrust]
    insights: List[str]
    fact_checks: Annotated[List[FactCheckResult], add]
    reflection: Optional[ReflectionResult]
    iteration: int
    chart_path: Optional[str]
    final_report: str

MAX_ITERATIONS = 2  # YOUR TODO: tune this for depth-vs-speed


## 9. Document ingestion helpers (for uploaded files)

In [ ]:
def ingest_documents(file_paths: List[str]) -> int:
    """Chunk-index local files into ChromaDB."""
    all_chunks = []
    for path in file_paths:
        try:
            if path.lower().endswith('.pdf'):
                docs = PyPDFLoader(path).load()
            else:
                docs = TextLoader(path).load()
            for d in docs:
                d.metadata["source"] = "user_upload"
                d.metadata["file"] = path
            all_chunks.extend(splitter.split_documents(docs))
        except Exception as e:
            console.print(f"  [yellow]Failed {path}:[/] {e}")
    if all_chunks:
        vector_store.add_documents(all_chunks)
        console.print(f"[green]Indexed {len(all_chunks)} chunks[/]")
    return len(all_chunks)


def upload_in_colab():
    """Trigger Colab file upload UI and ingest."""
    from google.colab import files
    uploaded = files.upload()
    return ingest_documents([f"/content/{n}" for n in uploaded.keys()])


def analyze_image(image_url: str, question: str) -> str:
    """Vision LLM call."""
    response = vision_llm.invoke([{"role": "user", "content": [
        {"type": "text", "text": question},
        {"type": "image_url", "image_url": {"url": image_url}},
    ]}])
    return response.content


## 10. The nine agents

Read the `# YOUR TODO` markers — those are the spots designed for your team's customization.

In [ ]:
# ========== 1. Planner ==========
def planner_agent(state: ResearchState) -> dict:
    structured_llm = reasoning_llm.with_structured_output(ResearchPlan)
    # YOUR TODO: customize this prompt for your domain (legal/medical/finance/etc.)
    prompt = f"""You are a research planner. Decompose this query into 3-5 focused sub-questions
and identify key concepts. Each sub-question should be independently answerable.

QUERY: {state['query']}"""
    plan = structured_llm.invoke(prompt)
    console.print(f"[bold cyan]Planner[/]: {len(plan.sub_questions)} sub-questions")
    return {"plan": plan, "iteration": 0}


# ========== 2. Retriever ==========
def retriever_agent(state: ResearchState) -> dict:
    findings, new_docs = [], []
    if state['iteration'] == 0:
        questions = [q.question for q in state['plan'].sub_questions]
    else:
        # On loop iterations, target gaps + overlooked angles from Reflection
        r = state.get('reflection')
        questions = (r.coverage_gaps + r.overlooked_angles) if r else []

    console.print(f"[bold cyan]Retriever[/] (iter {state['iteration']}): {len(questions)} questions")

    # YOUR TODO: smart tool routing — only call ArXiv for academic queries,
    # only Wikipedia for definitional ones, etc. Currently calls all 3 every time.
    for question in questions[:3]:
        try:
            for r in tavily_tool.invoke(question):
                content = r.get('content', '')
                findings.append(f"[Web] {content[:500]}")
                new_docs.append(Document(page_content=content,
                    metadata={"source": "tavily", "url": r.get('url', ''), "question": question}))
        except Exception as e:
            console.print(f"  [yellow]Tavily failed:[/] {e}")
        try:
            txt = arxiv_tool.invoke(question)
            findings.append(f"[ArXiv] {txt[:600]}")
            new_docs.append(Document(page_content=txt,
                metadata={"source": "arxiv", "question": question}))
        except Exception as e:
            console.print(f"  [yellow]ArXiv failed:[/] {e}")
        try:
            txt = wiki_tool.invoke(question)
            findings.append(f"[Wikipedia] {txt[:500]}")
            new_docs.append(Document(page_content=txt,
                metadata={"source": "wikipedia", "question": question}))
        except Exception as e:
            console.print(f"  [yellow]Wikipedia failed:[/] {e}")

    if new_docs:
        chunked = splitter.split_documents(new_docs)
        vector_store.add_documents(chunked)
        console.print(f"  Indexed {len(chunked)} chunks into RAG")
    return {"raw_findings": findings, "documents": new_docs}


# ========== 3. Source Reliability ==========
def source_reliability_agent(state: ResearchState) -> dict:
    """Rate each source by trust level."""
    if not state.get('documents'):
        return {"source_assessments": []}
    structured_llm = fast_llm.with_structured_output(SourceAssessment)

    items = [{
        "id": doc.metadata.get('url', doc.metadata.get('source', 'unknown')),
        "type": doc.metadata.get('source', 'unknown'),
        "snippet": doc.page_content[:200]
    } for doc in state['documents'][-10:]]
    sources_text = "\n".join([f"- [{s['type']}] {s['id']}: {s['snippet']}" for s in items])

    # YOUR TODO: customize trust criteria for your domain.
    # E.g. for medical: prioritize PubMed > NIH.gov > NEJM > general news.
    # For legal: prioritize SCOTUS > circuit courts > law reviews > blogs.
    prompt = f"""Rate each source by trustworthiness.
TRUST CRITERIA:
- HIGH: govt reports, peer-reviewed (arxiv), primary sources, established institutions
- MEDIUM: reputable news, established blogs, secondary analysis
- LOW: random blogs, social media, unsourced opinion

SOURCES:
{sources_text}"""
    result = structured_llm.invoke(prompt)
    high = sum(1 for a in result.assessments if a.trust_score == "high")
    console.print(f"[bold cyan]SourceReliability[/]: {high}/{len(result.assessments)} high-trust")
    return {"source_assessments": result.assessments}


# ========== 4. Multimodal ==========
def multimodal_agent(state: ResearchState) -> dict:
    """Analyze figures from any uploaded PDFs in our doc set."""
    import pypdf
    visual_findings = []
    seen = set()

    for doc in state.get('documents', []):
        path = doc.metadata.get('file', '')
        if path.endswith('.pdf') and path not in seen:
            seen.add(path)
            try:
                reader = pypdf.PdfReader(path)
                imgs = []
                for page in reader.pages:
                    for img in page.images:
                        imgs.append(img.data)
                        if len(imgs) >= 3:
                            break
                    if len(imgs) >= 3:
                        break
                for i, img_bytes in enumerate(imgs):
                    b64 = base64.b64encode(img_bytes).decode()
                    finding = analyze_image(
                        f"data:image/png;base64,{b64}",
                        f"Research query: {state['query']}\n\n"
                        "Describe this figure. Extract any quantitative data or trends. "
                        "If irrelevant to the query, say so briefly."
                    )
                    visual_findings.append(f"[Visual] {path} fig {i}: {finding}")
            except Exception as e:
                console.print(f"  [yellow]Multimodal:[/] {e}")

    # YOUR TODO: extend to extract images from web pages (BeautifulSoup +
    # download), or from URLs in Tavily results. Currently only handles PDFs.
    if visual_findings:
        console.print(f"[bold cyan]Multimodal[/]: {len(visual_findings)} visual findings")
    else:
        console.print(f"[bold cyan]Multimodal[/]: no images to analyze")
    return {"raw_findings": visual_findings}


# ========== 5. Analyzer ==========
def analyzer_agent(state: ResearchState) -> dict:
    findings_text = "\n\n".join(state['raw_findings'][-15:])
    # YOUR TODO: tune this prompt to extract domain-specific entity types.
    # E.g. for finance: "extract metrics, percentages, dollar amounts, dates."
    prompt = f"""Extract 3-5 key insights from these findings. Each insight must be
a concrete, evidence-backed claim — no fluff.

FINDINGS:
{findings_text}

Return as a numbered list, one insight per line."""
    response = fast_llm.invoke(prompt)
    insights = [i.strip().lstrip("0123456789.-) ") for i in response.content.split('\n')
                if i.strip() and len(i.strip()) > 20]
    console.print(f"[bold cyan]Analyzer[/]: {len(insights)} insights")
    return {"insights": insights}


# ========== 6. FactChecker (RAG) ==========
def fact_checker_agent(state: ResearchState) -> dict:
    structured_llm = reasoning_llm.with_structured_output(FactCheckResult)
    fact_checks = []
    for insight in state.get('insights', [])[:3]:
        relevant = vector_store.similarity_search(insight, k=3)
        evidence = "\n---\n".join([d.page_content[:300] for d in relevant])
        prompt = f"""Fact-check this claim. Be skeptical.

CLAIM: {insight}

EVIDENCE:
{evidence}

Verdict: 'supported', 'unsupported', or 'contradicted'."""
        try:
            fact_checks.append(structured_llm.invoke(prompt))
        except Exception as e:
            console.print(f"  [yellow]Fact-check:[/] {e}")
    console.print(f"[bold cyan]FactChecker[/]: {len(fact_checks)} claims checked")
    return {"fact_checks": fact_checks}


# ========== 7. Reflection (replaces basic GapFiller) ==========
def reflection_agent(state: ResearchState) -> dict:
    """Find gaps, contradictions, AND overlooked angles. Decide if loop fires."""
    structured_llm = reasoning_llm.with_structured_output(ReflectionResult)
    sub_qs = "\n".join([f"- {q.question}" for q in state['plan'].sub_questions])
    insights = "\n".join([f"- {i}" for i in state.get('insights', [])])
    fcs = "\n".join([f"- [{fc.verdict}] {fc.claim}" for fc in state.get('fact_checks', [])])

    prompt = f"""You are a critical reviewer. Reflect on this research.

ORIGINAL SUB-QUESTIONS:
{sub_qs}

CURRENT INSIGHTS:
{insights}

FACT-CHECK RESULTS:
{fcs}

Identify:
1. coverage_gaps — sub-questions still without strong evidence
2. contradictions — claims that disagree across sources (e.g. "EVs reduce emissions" vs "battery production increases emissions")
3. overlooked_angles — perspectives not yet researched (rural impact, opposing views, second-order effects)

Iteration {state['iteration']}/{MAX_ITERATIONS}. Set needs_more_research=True only if concrete
gaps or unresolved contradictions remain."""
    r = structured_llm.invoke(prompt)
    if state['iteration'] + 1 >= MAX_ITERATIONS:
        r.needs_more_research = False
    console.print(f"[bold cyan]Reflection[/]: gaps={len(r.coverage_gaps)}, "
                  f"contradictions={len(r.contradictions)}, overlooked={len(r.overlooked_angles)}")
    return {"reflection": r, "iteration": state['iteration'] + 1}


# ========== 8. Visualization ==========
def visualization_agent(state: ResearchState) -> dict:
    """Generate a chart from the insights if numeric data is present."""
    structured_llm = fast_llm.with_structured_output(ChartSpec)
    insights = "\n".join(state.get('insights', []))
    findings = "\n".join(state.get('raw_findings', [])[-10:])

    prompt = f"""Extract quantitative data from the research suitable for ONE chart.
Return chart_type='none' if no clean numeric comparison exists.

INSIGHTS: {insights}
FINDINGS: {findings[:2000]}"""
    spec = structured_llm.invoke(prompt)
    if spec.chart_type == "none" or not spec.y_values or len(spec.y_values) != len(spec.x_labels):
        console.print(f"[bold cyan]Visualization[/]: no chartable data")
        return {"chart_path": None}

    fig, ax = plt.subplots(figsize=(8, 5))
    if spec.chart_type == "bar":
        ax.bar(spec.x_labels, spec.y_values)
    elif spec.chart_type == "line":
        ax.plot(spec.x_labels, spec.y_values, marker='o')
    elif spec.chart_type == "pie":
        ax.pie(spec.y_values, labels=spec.x_labels, autopct='%1.1f%%')
    ax.set_title(spec.title)
    if spec.chart_type != "pie":
        ax.set_ylabel(spec.y_label)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    out = "/content/research_chart.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()

    # YOUR TODO: add multiple chart support, themed styling, or interactive Plotly.
    console.print(f"[bold cyan]Visualization[/]: chart saved → {out}")
    return {"chart_path": out}


# ========== 9. ReportBuilder ==========
def report_builder_agent(state: ResearchState) -> dict:
    insights = "\n".join([f"- {i}" for i in state.get('insights', [])])
    fact_checks = "\n".join([f"- **[{fc.verdict}]** {fc.claim} — {fc.evidence}"
                              for fc in state.get('fact_checks', [])])
    sources = sorted(set(d.metadata.get('source', '?') for d in state.get('documents', [])))

    # Trust summary from SourceReliability
    trust_counts = {"high": 0, "medium": 0, "low": 0}
    for a in state.get('source_assessments', []):
        trust_counts[a.trust_score] = trust_counts.get(a.trust_score, 0) + 1
    confidence = "High" if trust_counts["high"] >= 3 else                  "Medium" if trust_counts["high"] + trust_counts["medium"] >= 3 else "Low"

    # Contradictions from Reflection
    r = state.get('reflection')
    contradictions = "\n".join([f"- {c}" for c in (r.contradictions if r else [])]) or "_None identified_"

    chart_md = f"\n\n![Chart]({state['chart_path']})\n" if state.get('chart_path') else ""

    # YOUR TODO: customize report structure for your domain (medical paper format, legal brief, etc.)
    prompt = f"""Write a research report on: {state['query']}

KEY INSIGHTS:
{insights}

FACT-CHECKED CLAIMS:
{fact_checks}

CONTRADICTIONS FOUND:
{contradictions}

SOURCES: {', '.join(sources)} | Trust: {trust_counts}
CONFIDENCE: {confidence}
ITERATIONS: {state['iteration']}

Format as markdown with: ## Executive Summary, ## Key Findings, ## Evidence Assessment,
## Contradictions & Open Questions, ## Recommendations, ## Sources & Confidence.
End with: 'Confidence Score: {confidence} (based on {trust_counts["high"]} high-trust sources).'
Be concise but rigorous."""
    response = reasoning_llm.invoke(prompt)
    final = response.content + chart_md
    console.print("[bold green]ReportBuilder[/]: report ready ✅")
    return {"final_report": final}


## 11. Conditional routing

In [ ]:
def should_continue(state: ResearchState) -> Literal["retriever", "visualization"]:
    r = state.get('reflection')
    if r and r.needs_more_research and state['iteration'] < MAX_ITERATIONS:
        return "retriever"
    return "visualization"


## 12. Assemble the graph (with persistent memory)

`SqliteSaver` persists agent state across runs. Different `thread_id` values isolate different research projects.

In [ ]:
checkpointer = SqliteSaver.from_conn_string("./checkpoints.db")

graph = StateGraph(ResearchState)

graph.add_node("planner", planner_agent)
graph.add_node("retriever", retriever_agent)
graph.add_node("source_reliability", source_reliability_agent)
graph.add_node("multimodal", multimodal_agent)
graph.add_node("analyzer", analyzer_agent)
graph.add_node("fact_checker", fact_checker_agent)
graph.add_node("reflection", reflection_agent)
graph.add_node("visualization", visualization_agent)
graph.add_node("report_builder", report_builder_agent)

graph.add_edge(START, "planner")
graph.add_edge("planner", "retriever")
graph.add_edge("retriever", "source_reliability")
graph.add_edge("source_reliability", "multimodal")
graph.add_edge("multimodal", "analyzer")
graph.add_edge("analyzer", "fact_checker")
graph.add_edge("fact_checker", "reflection")
graph.add_conditional_edges("reflection", should_continue,
    {"retriever": "retriever", "visualization": "visualization"})
graph.add_edge("visualization", "report_builder")
graph.add_edge("report_builder", END)

app = graph.compile(checkpointer=checkpointer)
print("✅ Graph compiled with checkpointer (memory enabled)")


## 13. Render the graph

In [ ]:
from IPython.display import Image, display
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_ascii())


## 14. Run end-to-end (with thread_id for memory)

In [ ]:
# YOUR TODO: replace this with your team's signature demo query.
# Pick something that exercises all 9 agents — needs multi-source data, has
# legitimate contradictions, lends itself to a chart.
query = "How are GLP-1 drugs (like Ozempic) reshaping food and beverage company strategy in 2025?"

initial_state = {
    "query": query,
    "raw_findings": [], "documents": [], "fact_checks": [],
    "iteration": 0, "source_assessments": [],
}

# Same thread_id across calls = continuity. Change it for a fresh research session.
config = {"configurable": {"thread_id": "demo-session-1"}, "recursion_limit": 30}

# Stream for visibility — same role LangSmith would play
for step in app.stream(initial_state, config=config):
    for node, update in step.items():
        keys = list(update.keys()) if isinstance(update, dict) else []
        console.print(f"[bold magenta]→ {node}[/] updated: {keys}")

# Get final state
result = app.invoke(initial_state, config=config)
console.print(Markdown(result["final_report"]))


## 15. Memory in action

Same `thread_id` resumes from where you left off. Different `thread_id` = isolated session.

In [ ]:
# Inspect persisted state for a thread
state_snapshot = app.get_state({"configurable": {"thread_id": "demo-session-1"}})
print("Persisted state keys:", list(state_snapshot.values.keys()) if state_snapshot.values else "none")
print(f"Last iteration completed: {state_snapshot.values.get('iteration', 'n/a')}")

# Try a fresh thread for isolation:
# config2 = {"configurable": {"thread_id": "different-research-project"}}
# app.invoke({"query": "...different query..."}, config=config2)


---

# 16. YOUR TEAM'S SLOT — pick at least one to differentiate

The skeleton runs as-is and hits every scoring criterion the prior submissions covered. To win, your team needs to put a fingerprint on it. Pick at least one of these:

### Option A — Domain specialization (most defensible)
Pick a vertical and tune the system to it:
- **Legal**: swap Tavily for CourtListener API; add a `PrecedentRanker` agent
- **Medical**: swap ArXiv for PubMed E-utilities; add an `EvidenceGrader` agent
- **Finance**: add SEC EDGAR tool; add a `FinancialMetrics` agent that builds a DataFrame
- **Security**: swap for CVE/NVD APIs; add a `SeverityScorer` agent
- **Your domain**: swap one tool, add one expert agent, rewrite the Planner prompt

The graph topology stays identical. Five files change. Half a day of work. **This is what makes your demo land.**

### Option B — Add a custom agent
- **Critic agent** — argues against the report's conclusions, triggers a revision loop (Pydantic schema in earlier conversations; ask Claude for the stub)
- **CitationFormatter agent** — produces APA/MLA bibliography from documents
- **Comparison agent** — when query is "X vs Y", branches into parallel sub-graphs
- **Translator agent** — multi-language input/output

### Option C — Improve a stub
The TODO markers throughout the agent code are real. The biggest wins:
- **Smart tool routing** in Retriever (only call ArXiv for academic queries, etc.)
- **HTML image extraction** in Multimodal (currently PDF-only)
- **Domain-specific trust criteria** in SourceReliability
- **Multi-chart support** in Visualization (one bar + one line, dashboard-style)
- **Smart chunking** with metadata filtering for RAG

### Option D — Production hygiene (free points)
Score axis the prior submissions kept losing:
- `tests/test_agents.py` with mocked LLM responses for each agent
- `pyproject.toml` + pinned `requirements.txt`
- `.env.example`, proper `.gitignore`
- `Makefile` with `install`, `run`, `test`, `lint`
- README with the architecture diagram, demo gif, deployed Streamlit link

### Option E — Streamlit UI polish
A starter `app.py` is provided alongside this notebook. Make it look professional:
- File upload sidebar that calls `ingest_documents`
- Past sessions list (uses checkpointer threads)
- Live agent log streaming via `app.stream()`
- Chart rendering inline
- Confidence badge in the header

---

If your team has time for two: **Option A + Option D**. Domain specialization makes the demo memorable; production hygiene closes the gap that cost the most polished prior submission their position.